In [ ]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.inspection import PartialDependenceDisplay
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error 
import shap
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer 
from sklearn.ensemble import RandomForestRegressor



df = pd.read_csv("../datasets/winequality-red.csv")
print(df.isna().sum())
print(df.head())

X = df.drop("quality",axis=1)
y = df["quality"]

X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42)

model = XGBRegressor(
    n_estimators=100,
    random_state=42
)
model.fit(X_train,y_train)


for feature in X_train.columns:
    fig, ax = plt.subplots(figsize=(10,6))
    PartialDependenceDisplay.from_estimator(model, X=X_train, features=[feature], ax=ax)
    plt.show()

# volatile acidity --> inverse exponential; residual sugar --> logarithmic; density --> sine; sulphates --> logarithmic; alcohol --> logarithmic; 

def engineer_features(X):
    X = X.copy()
    X["volatile_acidity_exp"] = np.exp(-X["volatile acidity"])
    X["residual_sugar_log"] = np.log1p(X["residual sugar"])
    X["density_sin"] = np.sin(X["density"])
    X["sulphates_log"] = np.log1p(X["sulphates"])
    X["alcohol_log"] = np.log1p(X["alcohol"])
    return X

X_train_eng = engineer_features(X_train)

explainer = shap.Explainer(model)
shap_interaction_values = explainer.shap_interaction_values(X_train)

mean_abs_interactions = np.abs(shap_interaction_values).mean(axis=0)
feature_names = X_train.columns 
df_interactions = pd.DataFrame(mean_abs_interactions, columns = feature_names, index = feature_names)
df_interactions
upper_tri_indices = np.triu_indices_from(df_interactions, k=1)
top_3 = pd.DataFrame({
    'Feature_1': df_interactions.columns[upper_tri_indices[1]],
    'Feature_2': df_interactions.index[upper_tri_indices[0]],
    'Interaction_Strength': df_interactions.values[upper_tri_indices]
}).sort_values(by='Interaction_Strength', ascending=False).head(3)

def add_interactions(X):
    X=X.copy()
    for index,row in top_3.iterrows():
        f1 = row["Feature_1"]
        f2 = row["Feature_2"]
        X[f"{f1}_x_{f2}"] = X[f1] * X[f2]
    return X

eng_features = FunctionTransformer(engineer_features)
interaction_features = FunctionTransformer(add_interactions)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

pipeline_features = Pipeline([
    ("features", eng_features),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

pipeline_features_interactions = Pipeline([
    ("features", eng_features),
    ("interactions", interaction_features),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

models = [(pipeline, "Baseline"), (pipeline_features, "Engineered Features"), (pipeline_features_interactions, "Engineered + Interactions")]

cv_strategy = KFold(n_splits=5, shuffle=True, random_state = 42)

for i in models:
    scores = cross_val_score(i[0], X_train, y_train, cv=cv_strategy, scoring = "neg_mean_squared_error")
    print(f"{i[1]} MSE: {scores.mean()}")

#Streamlit

#pipeline_features achieves best cross-val score 



In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

pipeline_rf = Pipeline([
    ("features", eng_features),
    ("interactions", interaction_features),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor())
])


pipeline_xgb = Pipeline([
    ("features", eng_features),
    ("interactions", interaction_features),
    ("scaler", StandardScaler()),
    ("model", XGBRegressor())
])

xgb_param_grid ={
    'model__n_estimators': [100, 200, 300],      # Number of gradient boosted trees
    'model__learning_rate': [0.01, 0.05, 0.1],    # Step size shrinkage (eta)
    'model__max_depth': [3, 5, 7],                # Maximum tree depth for base learners
    'model__min_child_weight': [1, 3, 5],         # Minimum sum of instance weight needed in a child
    'model__subsample': [0.7, 0.9, 1.0],          # Subsample ratio of the training instances
    'model__colsample_bytree': [0.7, 0.9, 1.0],   # Subsample ratio of columns when constructing each tree
    'model__gamma': [0, 0.1, 0.2]                 # Minimum loss reduction required to make a further partition
}

rf_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2', None],
    'model__bootstrap': [True, False]
}

model_params = [(pipeline_xgb,xgb_param_grid, "XGB"), (pipeline_rf, rf_param_grid, "RF")]
best_params = []
for model, param_grid, name in model_params:
    grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, n_iter = 500, cv=cv_strategy, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    best_params.append((name, grid_search.best_params_, grid_search.best_score_))
    print(f"{name} Best Params: {grid_search.best_params_}, Best MSE: {grid_search.best_score_}")

In [13]:
import json
params = best_params[1][1]
params = {k.replace("model__", ""):v for k,v in params.items()}
with open("../model/params.json", "w") as pf:
    json.dump(params,pf)
